In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import random
import time
from collections import Counter

DRIVE_ROOT = Path("/kaggle/working/Drive")
DATA_ROOT = DRIVE_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed" / "bdd100k"

CONFIGS_ROOT = DRIVE_ROOT / "configs"

RUN_ROOT = DRIVE_ROOT / "runs" / "stage1" / "drive_yolo_nano_384"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
LOG_DIR = RUN_ROOT / "logs"
METRICS_DIR = RUN_ROOT / "metrics"
VIS_DIR = RUN_ROOT / "visualizations"

OUTPUT_EXPORT_ROOT = Path("/kaggle/working/drive-stage1-yolo-nano-384-output")

for p in [
    DRIVE_ROOT,
    DATA_ROOT,
    PROCESSED_ROOT,
    CONFIGS_ROOT,
    RUN_ROOT,
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    VIS_DIR,
    OUTPUT_EXPORT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

os.chdir(DRIVE_ROOT)

if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

print("Drive root:", DRIVE_ROOT)
print("Run root:", RUN_ROOT)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Output export root:", OUTPUT_EXPORT_ROOT)
print("Current working directory:", Path.cwd())

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("Torch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
    print("Torch compiled CUDA arch list:")
    print(torch.cuda.get_arch_list())

In [ ]:
import torch
import cv2
import numpy as np
import pandas as pd

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
CONFIG = {
    "project": "Drive",
    "stage": "Stage 1",
    "architecture": "Drive-YOLO-Nano-384",
    "description": "Nano P2-P5 + SPPF + BiFPN-lite road/lane/edge segmentation at 384x640",

    "image_height": 384,
    "image_width": 640,

    "use_p1": False,
    "use_sppf": True,
    "neck": "bifpn_lite_p2_p5",

    "backbone_channels": [16, 32, 64, 128, 256],
    "neck_channels": 48,

    "epochs": 50,

    "batch_size": 16,
    "grad_accum_steps": 1,

    "val_batch_size": 4,
    "fast_val_samples": 1000,
    "validate_every": 5,
    "run_full_val": False,

    "train_num_workers": 0,
    "val_num_workers": 0,
    "pin_memory": False,
    "persistent_workers": False,

    "lr": 7e-4,
    "weight_decay": 1e-4,
    "min_lr": 1e-6,

    "road_weight": 1.0,
    "lane_weight": 1.0,
    "edge_weight": 1.5,

    "lane_threshold": 0.35,
    "edge_threshold": 0.40,

    "amp": True,
    "validate_amp": True,

    "resume": False,

    "save_every": 5,
    "checkpoint_every_steps": 1000,

    "multi_gpu": True,
    "parallel_backend": "DataParallel",

    "seed": 42,
}

train_config_path = CONFIGS_ROOT / "stage1_drive_yolo_nano_384_train_config.json"
train_config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print(json.dumps(CONFIG, indent=2))
print("Saved:", train_config_path)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(f"  capability:", torch.cuda.get_device_capability(i))

multi_gpu_requested = globals().get("CONFIG", {}).get("multi_gpu", False)

if multi_gpu_requested:
    if torch.cuda.device_count() < 2:
        print("[warn] CONFIG requests multi-GPU, but fewer than 2 CUDA devices are visible.")
    else:
        print("Multi-GPU training available.")
else:
    print("CONFIG not defined yet or multi_gpu is disabled.")

In [ ]:
BDD_ROOT = Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k")
PROCESSED_DATASET_ROOT = Path(
    "/kaggle/input/datasets/dragver86/drive-bdd100k-processedh/drive-stage1-bdd100k-processed"
)

print("BDD_ROOT:", BDD_ROOT, "exists:", BDD_ROOT.exists())
print("PROCESSED_DATASET_ROOT:", PROCESSED_DATASET_ROOT, "exists:", PROCESSED_DATASET_ROOT.exists())

if not BDD_ROOT.exists():
    raise FileNotFoundError(f"Original BDD100K dataset not found: {BDD_ROOT}")

if not PROCESSED_DATASET_ROOT.exists():
    raise FileNotFoundError(f"Processed dataset not found: {PROCESSED_DATASET_ROOT}")

print("\n/kaggle/input contents:")
!ls -lah /kaggle/input

print("\nProcessed dataset preview:")
!find /kaggle/input/datasets/dragver86/drive-bdd100k-processedh/drive-stage1-bdd100k-processed -maxdepth 2 -type f | sort | head -80

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(path: Path) -> int:
    if path is None or not path.exists():
        return 0
    return sum(
        1 for p in path.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

def find_dirs_containing(root: Path, required_parts, must_have_images=True):
    required_parts = [x.lower() for x in required_parts]
    candidates = []

    for p in root.rglob("*"):
        if not p.is_dir():
            continue

        text = p.as_posix().lower()

        if all(part in text for part in required_parts):
            if must_have_images and count_images(p) == 0:
                continue
            candidates.append(p)

    return sorted(candidates, key=lambda x: (-count_images(x), len(x.parts)))

train_img_candidates = find_dirs_containing(BDD_ROOT, ["images", "100k", "train"])
val_img_candidates = find_dirs_containing(BDD_ROOT, ["images", "100k", "val"])

print("Train image candidates:")
for p in train_img_candidates[:5]:
    print(" ", p, "count:", count_images(p))

print("\nVal image candidates:")
for p in val_img_candidates[:5]:
    print(" ", p, "count:", count_images(p))

if not train_img_candidates:
    raise FileNotFoundError("Could not find BDD100K train image directory under /kaggle/input/bdd-100k.")

if not val_img_candidates:
    raise FileNotFoundError("Could not find BDD100K val image directory under /kaggle/input/bdd-100k.")

TRAIN_IMAGE_ROOT = train_img_candidates[0]
VAL_IMAGE_ROOT = val_img_candidates[0]

print("\nSelected train image root:", TRAIN_IMAGE_ROOT)
print("Train image count:", count_images(TRAIN_IMAGE_ROOT))

print("\nSelected val image root:", VAL_IMAGE_ROOT)
print("Val image count:", count_images(VAL_IMAGE_ROOT))

In [ ]:
required_paths = {
    "train portable manifest": PROCESSED_DATASET_ROOT / "manifests" / "manifest_train_portable.csv",
    "val portable manifest": PROCESSED_DATASET_ROOT / "manifests" / "manifest_val_portable.csv",

    "road train": PROCESSED_DATASET_ROOT / "road_masks" / "train",
    "road val": PROCESSED_DATASET_ROOT / "road_masks" / "val",

    "lane train": PROCESSED_DATASET_ROOT / "lane_masks" / "train",
    "lane val": PROCESSED_DATASET_ROOT / "lane_masks" / "val",

    "edge train": PROCESSED_DATASET_ROOT / "edge_masks" / "train",
    "edge val": PROCESSED_DATASET_ROOT / "edge_masks" / "val",
}

for name, path in required_paths.items():
    if path.is_dir():
        n = len(list(path.glob("*.png")))
        print(f"{name:28s} DIR  exists={path.exists()} count={n} path={path}")
    else:
        print(f"{name:28s} FILE exists={path.exists()} path={path}")

for name, path in required_paths.items():
    assert path.exists(), f"Missing required processed dataset path: {name}: {path}"

In [ ]:
TRAIN_PORTABLE = PROCESSED_DATASET_ROOT / "manifests" / "manifest_train_portable.csv"
VAL_PORTABLE = PROCESSED_DATASET_ROOT / "manifests" / "manifest_val_portable.csv"

def build_image_index(image_root: Path):
    index = {}
    duplicates = Counter()

    for p in image_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            if p.name in index:
                duplicates[p.name] += 1
                continue
            index[p.name] = p

    if duplicates:
        print("[warn] Duplicate image basenames:", len(duplicates))

    return index

def build_absolute_manifest(
    portable_csv: Path,
    out_csv: Path,
    image_root: Path,
    processed_root: Path,
):
    df = pd.read_csv(portable_csv)
    image_index = build_image_index(image_root)

    rows = []
    missing_images = 0
    missing_masks = 0

    for _, row in df.iterrows():
        image_name = row["image_name"]
        image_path = image_index.get(image_name)

        road_mask_path = processed_root / row["road_mask_relpath"]
        lane_mask_path = processed_root / row["lane_mask_relpath"]
        edge_mask_path = processed_root / row["edge_mask_relpath"]

        if image_path is None:
            missing_images += 1
            continue

        if not road_mask_path.exists() or not lane_mask_path.exists() or not edge_mask_path.exists():
            missing_masks += 1
            continue

        rows.append(
            {
                "image_id": row["image_id"],
                "image_name": image_name,
                "image_path": str(image_path),
                "road_mask_path": str(road_mask_path),
                "lane_mask_path": str(lane_mask_path),
                "edge_mask_path": str(edge_mask_path),
                "split": row["split"],
                "height": int(row["height"]),
                "width": int(row["width"]),
                "has_road": int(row["has_road"]),
                "has_lane": int(row["has_lane"]),
                "road_pixels": int(row["road_pixels"]),
                "lane_pixels": int(row["lane_pixels"]),
                "edge_pixels": int(row["edge_pixels"]),
            }
        )

    out = pd.DataFrame(rows)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_csv, index=False)

    print("Wrote:", out_csv)
    print("Rows:", len(out))
    print("Missing images:", missing_images)
    print("Missing masks:", missing_masks)

    return out

train_manifest = PROCESSED_ROOT / "manifest_train.csv"
val_manifest = PROCESSED_ROOT / "manifest_val.csv"

train_df = build_absolute_manifest(
    portable_csv=TRAIN_PORTABLE,
    out_csv=train_manifest,
    image_root=TRAIN_IMAGE_ROOT,
    processed_root=PROCESSED_DATASET_ROOT,
)

val_df = build_absolute_manifest(
    portable_csv=VAL_PORTABLE,
    out_csv=val_manifest,
    image_root=VAL_IMAGE_ROOT,
    processed_root=PROCESSED_DATASET_ROOT,
)

print("\nTrain rows:", len(train_df))
print("Val rows:", len(val_df))

print("\nTrain head:")
print(train_df.head())

assert len(train_df) > 60000, f"Too few train rows: {len(train_df)}"
assert len(val_df) >= 9000, f"Too few val rows: {len(val_df)}"

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, Subset

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

class DriveTrainDataset(Dataset):
    def __init__(
        self,
        manifest_path: Path,
        image_height: int,
        image_width: int,
        train: bool = True,
    ):
        self.df = pd.read_csv(manifest_path).reset_index(drop=True)
        self.image_height = image_height
        self.image_width = image_width
        self.train = train
        self.transforms = self.build_transforms(train=train)

    def build_transforms(self, train: bool):
        if train:
            return A.Compose(
                [
                    A.Resize(
                        height=self.image_height,
                        width=self.image_width,
                        interpolation=cv2.INTER_LINEAR,
                        mask_interpolation=cv2.INTER_NEAREST,
                    ),
                    A.HorizontalFlip(p=0.5),
                    A.RandomBrightnessContrast(
                        brightness_limit=0.20,
                        contrast_limit=0.20,
                        p=0.35,
                    ),
                    A.HueSaturationValue(
                        hue_shift_limit=10,
                        sat_shift_limit=25,
                        val_shift_limit=20,
                        p=0.25,
                    ),
                    A.MotionBlur(blur_limit=5, p=0.15),
                    A.GaussianBlur(blur_limit=(3, 3), p=0.10),
                    A.Affine(
                        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                        scale=(0.90, 1.10),
                        rotate=(-3, 3),
                        interpolation=cv2.INTER_LINEAR,
                        mask_interpolation=cv2.INTER_NEAREST,
                        border_mode=cv2.BORDER_CONSTANT,
                        fill=0,
                        fill_mask=0,
                        p=0.35,
                    ),
                    A.Normalize(
                        mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225],
                    ),
                    ToTensorV2(),
                ],
                additional_targets={
                    "road_mask": "mask",
                    "lane_mask": "mask",
                    "edge_mask": "mask",
                },
            )

        return A.Compose(
            [
                A.Resize(
                    height=self.image_height,
                    width=self.image_width,
                    interpolation=cv2.INTER_LINEAR,
                    mask_interpolation=cv2.INTER_NEAREST,
                ),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
                ToTensorV2(),
            ],
            additional_targets={
                "road_mask": "mask",
                "lane_mask": "mask",
                "edge_mask": "mask",
            },
        )

    @staticmethod
    def read_rgb(path):
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    @staticmethod
    def read_mask(path):
        m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(path)
        return (m > 127).astype(np.uint8)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = self.read_rgb(row["image_path"])
        road = self.read_mask(row["road_mask_path"])
        lane = self.read_mask(row["lane_mask_path"])
        edge = self.read_mask(row["edge_mask_path"])

        if image.shape[:2] != road.shape[:2]:
            image = cv2.resize(
                image,
                (road.shape[1], road.shape[0]),
                interpolation=cv2.INTER_LINEAR,
            )

        transformed = self.transforms(
            image=image,
            road_mask=road,
            lane_mask=lane,
            edge_mask=edge,
        )

        image = transformed["image"].float()
        road = transformed["road_mask"].long()
        lane = transformed["lane_mask"].long()
        edge = transformed["edge_mask"].float().unsqueeze(0)

        return {
            "image": image,
            "road_mask": road,
            "lane_mask": lane,
            "edge_mask": edge,
            "image_id": row["image_id"],
        }

def drive_collate(batch):
    return {
        "image": torch.stack([b["image"] for b in batch], dim=0),
        "road_mask": torch.stack([b["road_mask"] for b in batch], dim=0),
        "lane_mask": torch.stack([b["lane_mask"] for b in batch], dim=0),
        "edge_mask": torch.stack([b["edge_mask"] for b in batch], dim=0),
        "image_id": [b["image_id"] for b in batch],
    }

train_ds = DriveTrainDataset(
    manifest_path=train_manifest,
    image_height=CONFIG["image_height"],
    image_width=CONFIG["image_width"],
    train=True,
)

val_ds = DriveTrainDataset(
    manifest_path=val_manifest,
    image_height=CONFIG["image_height"],
    image_width=CONFIG["image_width"],
    train=False,
)

rng = np.random.default_rng(CONFIG["seed"])
fast_val_n = min(CONFIG["fast_val_samples"], len(val_ds))
fast_val_indices = rng.choice(len(val_ds), size=fast_val_n, replace=False).tolist()
fast_val_ds = Subset(val_ds, fast_val_indices)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=CONFIG["train_num_workers"],
    pin_memory=torch.cuda.is_available(),
    persistent_workers=False,
    collate_fn=drive_collate,
)

fast_val_loader = DataLoader(
    fast_val_ds,
    batch_size=CONFIG["val_batch_size"],
    shuffle=False,
    drop_last=False,
    num_workers=CONFIG["val_num_workers"],
    pin_memory=torch.cuda.is_available(),
    collate_fn=drive_collate,
)

full_val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["val_batch_size"],
    shuffle=False,
    drop_last=False,
    num_workers=CONFIG["val_num_workers"],
    pin_memory=torch.cuda.is_available(),
    collate_fn=drive_collate,
)

print("Train dataset:", len(train_ds))
print("Val dataset:", len(val_ds))
print("Fast val dataset:", len(fast_val_ds))

print("Train batch size:", CONFIG["batch_size"])
print("Val batch size:", CONFIG["val_batch_size"])

print("Train batches:", len(train_loader))
print("Fast val batches:", len(fast_val_loader))
print("Full val batches:", len(full_val_loader))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def autopad(k, p=None):
    return k // 2 if p is None else p

class ConvBNAct(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()

        self.conv = nn.Conv2d(
            c1,
            c2,
            kernel_size=k,
            stride=s,
            padding=autopad(k),
            groups=g,
            bias=False,
        )
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class Bottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5):
        super().__init__()

        hidden = int(c2 * e)

        self.cv1 = ConvBNAct(c1, hidden, k=1, s=1)
        self.cv2 = ConvBNAct(hidden, c2, k=3, s=1)

        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.add else y

class C2f(nn.Module):
    def __init__(self, c1, c2, n=1, e=0.5):
        super().__init__()

        hidden = int(c2 * e)

        self.cv1 = ConvBNAct(c1, 2 * hidden, k=1, s=1)

        self.blocks = nn.ModuleList(
            [
                Bottleneck(
                    hidden,
                    hidden,
                    shortcut=True,
                    e=1.0,
                )
                for _ in range(n)
            ]
        )

        self.cv2 = ConvBNAct((2 + n) * hidden, c2, k=1, s=1)

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, dim=1))

        for block in self.blocks:
            y.append(block(y[-1]))

        return self.cv2(torch.cat(y, dim=1))

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()

        hidden = max(c1 // 2, 8)

        self.cv1 = ConvBNAct(c1, hidden, k=1, s=1)
        self.pool = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)
        self.cv2 = ConvBNAct(hidden * 4, c2, k=1, s=1)

    def forward(self, x):
        x = self.cv1(x)

        y1 = self.pool(x)
        y2 = self.pool(y1)
        y3 = self.pool(y2)

        return self.cv2(torch.cat([x, y1, y2, y3], dim=1))

class WeightedFusion(nn.Module):

    def __init__(self, n_inputs, eps=1e-4):
        super().__init__()

        self.weights = nn.Parameter(torch.ones(n_inputs, dtype=torch.float32))
        self.eps = eps

    def forward(self, inputs):
        weights = F.relu(self.weights)
        weights = weights / (weights.sum() + self.eps)

        out = 0.0
        for w, x in zip(weights, inputs):
            out = out + w * x

        return out


class DriveYOLONanoBackbone(nn.Module):

    def __init__(self, channels=(16, 32, 64, 128, 256)):
        super().__init__()

        c1, c2, c3, c4, c5 = channels

        self.stem = ConvBNAct(3, c1, k=3, s=2)

        self.stage2 = nn.Sequential(
            ConvBNAct(c1, c2, k=3, s=2),
            C2f(c2, c2, n=1, e=0.5),
        )

        self.stage3 = nn.Sequential(
            ConvBNAct(c2, c3, k=3, s=2),
            C2f(c3, c3, n=1, e=0.5),
        )

        self.stage4 = nn.Sequential(
            ConvBNAct(c3, c4, k=3, s=2),
            C2f(c4, c4, n=1, e=0.5),
        )

        self.stage5 = nn.Sequential(
            ConvBNAct(c4, c5, k=3, s=2),
            C2f(c5, c5, n=1, e=0.5),
        )

        self.sppf = SPPF(c5, c5, k=5)

    def forward(self, x):
        x = self.stem(x)

        p2 = self.stage2(x)
        p3 = self.stage3(p2)
        p4 = self.stage4(p3)
        p5 = self.stage5(p4)
        p5 = self.sppf(p5)

        return {
            "p2": p2,
            "p3": p3,
            "p4": p4,
            "p5": p5,
        }


class BiFPNLiteP2P5(nn.Module):

    def __init__(self, in_channels=(32, 64, 128, 256), out_channels=48):
        super().__init__()

        p2c, p3c, p4c, p5c = in_channels
        c = out_channels

        self.reduce_p2 = ConvBNAct(p2c, c, k=1, s=1)
        self.reduce_p3 = ConvBNAct(p3c, c, k=1, s=1)
        self.reduce_p4 = ConvBNAct(p4c, c, k=1, s=1)
        self.reduce_p5 = ConvBNAct(p5c, c, k=1, s=1)

        self.fuse_td4 = WeightedFusion(2)
        self.fuse_td3 = WeightedFusion(2)
        self.fuse_td2 = WeightedFusion(2)

        self.refine_td4 = ConvBNAct(c, c, k=3, s=1)
        self.refine_td3 = ConvBNAct(c, c, k=3, s=1)
        self.refine_td2 = ConvBNAct(c, c, k=3, s=1)

        self.down_2_to_3 = ConvBNAct(c, c, k=3, s=2)
        self.down_3_to_4 = ConvBNAct(c, c, k=3, s=2)
        self.down_4_to_5 = ConvBNAct(c, c, k=3, s=2)

        self.fuse_bu3 = WeightedFusion(3)
        self.fuse_bu4 = WeightedFusion(3)
        self.fuse_bu5 = WeightedFusion(2)

        self.refine_bu3 = ConvBNAct(c, c, k=3, s=1)
        self.refine_bu4 = ConvBNAct(c, c, k=3, s=1)
        self.refine_bu5 = ConvBNAct(c, c, k=3, s=1)

        self.final_fusion = WeightedFusion(4)
        self.final_refine = ConvBNAct(c, c, k=3, s=1)

        self.out_channels = c

    @staticmethod
    def up_to(x, ref):
        return F.interpolate(
            x,
            size=ref.shape[-2:],
            mode="nearest",
        )

    def forward(self, feats):
        p2 = self.reduce_p2(feats["p2"])
        p3 = self.reduce_p3(feats["p3"])
        p4 = self.reduce_p4(feats["p4"])
        p5 = self.reduce_p5(feats["p5"])

        td5 = p5

        td4 = self.fuse_td4([p4, self.up_to(td5, p4)])
        td4 = self.refine_td4(td4)

        td3 = self.fuse_td3([p3, self.up_to(td4, p3)])
        td3 = self.refine_td3(td3)

        td2 = self.fuse_td2([p2, self.up_to(td3, p2)])
        td2 = self.refine_td2(td2)

        bu2 = td2

        bu3 = self.fuse_bu3([p3, td3, self.down_2_to_3(bu2)])
        bu3 = self.refine_bu3(bu3)

        bu4 = self.fuse_bu4([p4, td4, self.down_3_to_4(bu3)])
        bu4 = self.refine_bu4(bu4)

        bu5 = self.fuse_bu5([p5, self.down_4_to_5(bu4)])
        bu5 = self.refine_bu5(bu5)

        fused = self.final_fusion(
            [
                bu2,
                self.up_to(bu3, bu2),
                self.up_to(bu4, bu2),
                self.up_to(bu5, bu2),
            ]
        )

        fused = self.final_refine(fused)

        return {
            "fused": fused,
            "p2": bu2,
            "p3": bu3,
            "p4": bu4,
            "p5": bu5,
        }


class SegHead(nn.Module):

    def __init__(self, c1, hidden, out_channels):
        super().__init__()

        self.net = nn.Sequential(
            ConvBNAct(c1, hidden, k=3, s=1),
            ConvBNAct(hidden, hidden, k=3, s=1),
            nn.Conv2d(hidden, out_channels, kernel_size=1),
        )

    def forward(self, x, output_size):
        x = self.net(x)

        return F.interpolate(
            x,
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )


class DriveYOLONano384(nn.Module):
    def __init__(
        self,
        backbone_channels=(16, 32, 64, 128, 256),
        neck_channels=48,
    ):
        super().__init__()

        self.backbone = DriveYOLONanoBackbone(
            channels=tuple(backbone_channels)
        )

        self.neck = BiFPNLiteP2P5(
            in_channels=tuple(backbone_channels[1:]),
            out_channels=neck_channels,
        )

        self.road_head = SegHead(
            c1=neck_channels,
            hidden=neck_channels,
            out_channels=2,
        )

        self.lane_head = SegHead(
            c1=neck_channels,
            hidden=neck_channels,
            out_channels=2,
        )

        self.edge_head = SegHead(
            c1=neck_channels,
            hidden=max(24, neck_channels // 2),
            out_channels=1,
        )

    def forward(self, x):
        output_size = x.shape[-2:]

        feats = self.backbone(x)
        neck_out = self.neck(feats)
        fused = neck_out["fused"]

        return {
            "road_logits": self.road_head(fused, output_size),
            "lane_logits": self.lane_head(fused, output_size),
            "edge_logits": self.edge_head(fused, output_size),
        }

try:
    del model
    torch.cuda.empty_cache()
except Exception:
    pass

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

core_model = DriveYOLONano384(
    backbone_channels=CONFIG["backbone_channels"],
    neck_channels=CONFIG["neck_channels"],
).to(device)

num_gpus = torch.cuda.device_count()

use_data_parallel = (
    CONFIG.get("multi_gpu", False)
    and torch.cuda.is_available()
    and num_gpus >= 2
)

if use_data_parallel:
    print(f"Using DataParallel on {num_gpus} GPUs.")
    model = nn.DataParallel(core_model, device_ids=list(range(num_gpus)))
else:
    print("Using single-device model.")
    model = core_model

model = model.to(device)

params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Architecture:", CONFIG["architecture"])
print("Primary device:", device)
print("Visible GPUs:", num_gpus)
print(f"Total parameters:     {params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Approx fp32 size:     {params * 4 / 1024**2:.2f} MB")
print(f"Approx fp16 size:     {params * 2 / 1024**2:.2f} MB")

In [ ]:
model.eval()

dummy = torch.randn(
    2,
    3,
    CONFIG["image_height"],
    CONFIG["image_width"],
    device=device,
)

with torch.no_grad():
    out = model(dummy)

print("road_logits:", out["road_logits"].shape)
print("lane_logits:", out["lane_logits"].shape)
print("edge_logits:", out["edge_logits"].shape)

assert out["road_logits"].shape == (2, 2, CONFIG["image_height"], CONFIG["image_width"])
assert out["lane_logits"].shape == (2, 2, CONFIG["image_height"], CONFIG["image_width"])
assert out["edge_logits"].shape == (2, 1, CONFIG["image_height"], CONFIG["image_width"])

del dummy, out
torch.cuda.empty_cache()

print("Drive-YOLO-Nano-384 forward pass successful.")

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, probs, target):
        probs = probs.float()
        target = target.float()

        intersection = (probs * target).sum(dim=(1, 2))
        union = probs.sum(dim=(1, 2)) + target.sum(dim=(1, 2))

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

class FocalCELoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction="none")
        pt = torch.exp(-ce)
        loss = self.alpha * (1.0 - pt) ** self.gamma * ce
        return loss.mean()

class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()

        road_weights = torch.tensor([0.40, 1.00], dtype=torch.float32)
        self.register_buffer("road_weights", road_weights)

        self.dice = DiceLoss()
        self.lane_focal = FocalCELoss(alpha=0.75, gamma=2.0)

    def forward(self, outputs, batch):
        road_logits = outputs["road_logits"]
        lane_logits = outputs["lane_logits"]
        edge_logits = outputs["edge_logits"]

        road_target = batch["road_mask"]
        lane_target = batch["lane_mask"]
        edge_target = batch["edge_mask"]

        road_ce = F.cross_entropy(
            road_logits,
            road_target,
            weight=self.road_weights.to(road_logits.device),
        )

        road_prob = F.softmax(road_logits, dim=1)[:, 1]
        road_dice = self.dice(road_prob, road_target)

        lane_focal = self.lane_focal(lane_logits, lane_target)
        lane_prob = F.softmax(lane_logits, dim=1)[:, 1]
        lane_dice = self.dice(lane_prob, lane_target)

        pos_weight = torch.tensor([4.0], device=edge_logits.device)
        edge_bce = F.binary_cross_entropy_with_logits(
            edge_logits,
            edge_target,
            pos_weight=pos_weight,
        )

        edge_prob = torch.sigmoid(edge_logits).squeeze(1)
        edge_dice = self.dice(edge_prob, edge_target.squeeze(1))

        road_loss = road_ce + road_dice
        lane_loss = lane_focal + lane_dice
        edge_loss = edge_bce + edge_dice

        total = (
            CONFIG["road_weight"] * road_loss
            + CONFIG["lane_weight"] * lane_loss
            + CONFIG["edge_weight"] * edge_loss
        )

        return total, {
            "loss_total": float(total.detach().cpu()),
            "loss_road": float(road_loss.detach().cpu()),
            "loss_lane": float(lane_loss.detach().cpu()),
            "loss_edge": float(edge_loss.detach().cpu()),
        }

@torch.no_grad()
def binary_counts(pred, target):
    pred = pred.bool()
    target = target.bool()

    tp = (pred & target).sum().item()
    fp = (pred & ~target).sum().item()
    fn = (~pred & target).sum().item()

    return tp, fp, fn

@torch.no_grad()
def metric_counts(outputs, batch):
    road_pred = torch.argmax(outputs["road_logits"], dim=1)

    lane_prob = F.softmax(outputs["lane_logits"], dim=1)[:, 1]
    edge_prob = torch.sigmoid(outputs["edge_logits"]).squeeze(1)

    lane_pred = lane_prob > CONFIG["lane_threshold"]
    edge_pred = edge_prob > CONFIG["edge_threshold"]

    road_target = batch["road_mask"]
    lane_target = batch["lane_mask"]
    edge_target = batch["edge_mask"].squeeze(1) > 0.5

    road_tp, road_fp, road_fn = binary_counts(road_pred == 1, road_target == 1)
    lane_tp, lane_fp, lane_fn = binary_counts(lane_pred, lane_target == 1)
    edge_tp, edge_fp, edge_fn = binary_counts(edge_pred, edge_target)

    return {
        "road_tp": road_tp,
        "road_fp": road_fp,
        "road_fn": road_fn,
        "lane_tp": lane_tp,
        "lane_fp": lane_fp,
        "lane_fn": lane_fn,
        "edge_tp": edge_tp,
        "edge_fp": edge_fp,
        "edge_fn": edge_fn,
    }

def compute_epoch_metrics(counts):
    eps = 1e-7

    road_iou = counts["road_tp"] / (
        counts["road_tp"] + counts["road_fp"] + counts["road_fn"] + eps
    )

    lane_precision = counts["lane_tp"] / (counts["lane_tp"] + counts["lane_fp"] + eps)
    lane_recall = counts["lane_tp"] / (counts["lane_tp"] + counts["lane_fn"] + eps)
    lane_f1 = 2 * lane_precision * lane_recall / (lane_precision + lane_recall + eps)

    edge_precision = counts["edge_tp"] / (counts["edge_tp"] + counts["edge_fp"] + eps)
    edge_recall = counts["edge_tp"] / (counts["edge_tp"] + counts["edge_fn"] + eps)
    edge_f1 = 2 * edge_precision * edge_recall / (edge_precision + edge_recall + eps)

    return {
        "road_iou": road_iou,
        "lane_precision": lane_precision,
        "lane_recall": lane_recall,
        "lane_f1": lane_f1,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,
    }

criterion = MultiTaskLoss().to(device)

In [ ]:
def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def get_clean_state_dict(model):
    return unwrap_model(model).state_dict()


def load_clean_state_dict(model, state_dict, strict=True):
    target_model = unwrap_model(model)

    cleaned = {}

    for k, v in state_dict.items():
        if k.startswith("module."):
            cleaned[k[len("module."):]] = v
        else:
            cleaned[k] = v

    target_model.load_state_dict(cleaned, strict=strict)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=CONFIG["min_lr"],
)

scaler = torch.cuda.amp.GradScaler(
    enabled=CONFIG["amp"] and device.type == "cuda"
)

start_epoch = 1
best_lane_f1 = -1.0
history = []

local_last_ckpt = CHECKPOINT_DIR / "last.pt"

external_resume_candidates = [
    Path("/kaggle/input/drive-stage1-yolo-nano-384-output/checkpoints/last.pt"),
    Path("/kaggle/input/drive-stage1-yolo-s-output/checkpoints/last.pt"),
]

resume_path = None

if CONFIG["resume"] and local_last_ckpt.exists():
    resume_path = local_last_ckpt
else:
    for candidate in external_resume_candidates:
        if CONFIG["resume"] and candidate.exists():
            resume_path = candidate
            break

if resume_path is not None:
    print("Resuming from:", resume_path)

    ckpt = torch.load(resume_path, map_location=device)

    load_clean_state_dict(model, ckpt["model"], strict=True)

    if "optimizer" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
            print("Loaded optimizer state.")
        except Exception as e:
            print("[warn] Could not load optimizer state:", repr(e))

    if "scheduler" in ckpt:
        try:
            scheduler.load_state_dict(ckpt["scheduler"])
            print("Loaded scheduler state.")
        except Exception as e:
            print("[warn] Could not load scheduler state:", repr(e))

    if "scaler" in ckpt:
        try:
            scaler.load_state_dict(ckpt["scaler"])
            print("Loaded AMP scaler state.")
        except Exception as e:
            print("[warn] Could not load scaler state:", repr(e))

    start_epoch = int(ckpt.get("epoch", 0)) + 1
    history = ckpt.get("history", [])
    best_lane_f1 = float(ckpt.get("best_lane_f1", -1.0))

    print("Resume start_epoch:", start_epoch)
    print("Resume best_lane_f1:", best_lane_f1)

else:
    print("Starting fresh training.")

In [ ]:
from tqdm import tqdm
import gc

def move_batch_to_device(batch, device):
    return {
        "image": batch["image"].to(device, non_blocking=True),
        "road_mask": batch["road_mask"].to(device, non_blocking=True),
        "lane_mask": batch["lane_mask"].to(device, non_blocking=True),
        "edge_mask": batch["edge_mask"].to(device, non_blocking=True),
        "image_id": batch["image_id"],
    }

def train_one_epoch(model, loader, optimizer, scaler, criterion, device, epoch):
    model.train()

    running = Counter()
    n_batches = 0

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(
        loader,
        desc=f"train epoch {epoch}",
        mininterval=30,
        leave=False,
    )

    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch, device)

        with torch.cuda.amp.autocast(enabled=CONFIG["amp"] and device.type == "cuda"):
            outputs = model(batch["image"])
            loss, loss_items = criterion(outputs, batch)
            loss_for_backward = loss / CONFIG["grad_accum_steps"]

        scaler.scale(loss_for_backward).backward()

        should_step = (
            step % CONFIG["grad_accum_steps"] == 0
            or step == len(loader)
        )

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        for k, v in loss_items.items():
            running[k] += v

        n_batches += 1

        if step % 200 == 0:
            pbar.set_postfix(
                {
                    "loss": running["loss_total"] / n_batches,
                    "road": running["loss_road"] / n_batches,
                    "lane": running["loss_lane"] / n_batches,
                    "edge": running["loss_edge"] / n_batches,
                }
            )

        del batch, outputs, loss

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {k: running[k] / max(1, n_batches) for k in running}


def validate(model, loader, criterion, device, epoch, label="fast_val"):
    model.eval()

    running_loss = Counter()
    counts = Counter()
    n_batches = 0

    pbar = tqdm(
        loader,
        desc=f"{label} epoch {epoch}",
        mininterval=30,
        leave=False,
    )

    with torch.inference_mode():
        for step, batch in enumerate(pbar, start=1):
            batch = move_batch_to_device(batch, device)

            with torch.cuda.amp.autocast(
                enabled=CONFIG.get("validate_amp", True) and device.type == "cuda"
            ):
                outputs = model(batch["image"])
                loss, loss_items = criterion(outputs, batch)

            batch_counts = metric_counts(outputs, batch)

            for k, v in loss_items.items():
                running_loss[k] += v

            for k, v in batch_counts.items():
                counts[k] += v

            n_batches += 1

            if step % 100 == 0:
                metrics_now = compute_epoch_metrics(counts)
                pbar.set_postfix(
                    {
                        "loss": running_loss["loss_total"] / n_batches,
                        "road_iou": metrics_now["road_iou"],
                        "lane_f1": metrics_now["lane_f1"],
                        "edge_f1": metrics_now["edge_f1"],
                    }
                )

            del batch, outputs, loss

    epoch_metrics = compute_epoch_metrics(counts)

    out = {}
    out.update({f"{label}/{k}": running_loss[k] / max(1, n_batches) for k in running_loss})
    out.update({f"{label}/{k}": v for k, v in epoch_metrics.items()})
    out[f"{label}/n_batches"] = n_batches

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out

In [ ]:
import gc
import time
import json

start_time = time.time()

if start_epoch > CONFIG["epochs"]:
    print(
        f"Checkpoint epoch {start_epoch - 1} is already >= configured epochs {CONFIG['epochs']}."
    )
    print("Increase CONFIG['epochs'] if you want to continue further.")

for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    train_log = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        criterion=criterion,
        device=device,
        epoch=epoch,
    )

    scheduler.step()

    preval_ckpt = {
        "epoch": epoch,
        "model": get_clean_state_dict(model),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "config": CONFIG,
        "history": history,
        "best_lane_f1": best_lane_f1,
        "status": "trained_before_validation",
        "parallel_backend": "DataParallel" if isinstance(model, nn.DataParallel) else "single",
        "num_gpus": torch.cuda.device_count(),
    }

    torch.save(preval_ckpt, CHECKPOINT_DIR / "last_preval.pt")
    torch.save(preval_ckpt, CHECKPOINT_DIR / f"epoch_{epoch:03d}_preval.pt")
    print(f"Saved pre-validation checkpoint for epoch {epoch}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    fast_val_log = validate(
        model=model,
        loader=fast_val_loader,
        criterion=criterion,
        device=device,
        epoch=epoch,
        label="fast_val",
    )

    should_full_val = (
        CONFIG.get("run_full_val", False)
        and "full_val_loader" in globals()
        and (
            epoch % CONFIG.get("validate_every", 5) == 0
            or epoch == CONFIG["epochs"]
        )
    )

    full_val_log = {}

    if should_full_val:
        print(f"Running full validation at epoch {epoch}")
        full_val_log = validate(
            model=model,
            loader=full_val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
            label="full_val",
        )

    epoch_log = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "architecture": CONFIG["architecture"],
        "parallel_backend": "DataParallel" if isinstance(model, nn.DataParallel) else "single",
        "num_gpus": torch.cuda.device_count(),
        **{f"train/{k}": v for k, v in train_log.items()},
        **fast_val_log,
        **full_val_log,
    }

    history.append(epoch_log)

    print(json.dumps(epoch_log, indent=2))

    current_lane_f1 = epoch_log.get(
        "full_val/lane_f1",
        epoch_log.get("fast_val/lane_f1", 0.0),
    )

    best_metric_source = (
        "full_val/lane_f1"
        if "full_val/lane_f1" in epoch_log
        else "fast_val/lane_f1"
    )

    last_ckpt = {
        "epoch": epoch,
        "model": get_clean_state_dict(model),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "config": CONFIG,
        "history": history,
        "best_lane_f1": best_lane_f1,
        "status": "validated",
        "parallel_backend": "DataParallel" if isinstance(model, nn.DataParallel) else "single",
        "num_gpus": torch.cuda.device_count(),
    }

    torch.save(last_ckpt, CHECKPOINT_DIR / "last.pt")

    if epoch % CONFIG.get("save_every", 5) == 0:
        torch.save(last_ckpt, CHECKPOINT_DIR / f"epoch_{epoch:03d}.pt")

    if current_lane_f1 > best_lane_f1:
        best_lane_f1 = current_lane_f1

        best_ckpt = {
            **last_ckpt,
            "best_lane_f1": best_lane_f1,
            "best_metric_source": best_metric_source,
        }

        torch.save(best_ckpt, CHECKPOINT_DIR / "best.pt")
        print(f"Saved best checkpoint: lane_f1={best_lane_f1:.6f}")

    history_path = METRICS_DIR / "history.json"
    history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed = time.time() - start_time

best_metrics = {
    "architecture": CONFIG["architecture"],
    "parallel_backend": "DataParallel" if isinstance(model, nn.DataParallel) else "single",
    "num_gpus": torch.cuda.device_count(),
    "best_lane_f1": best_lane_f1,
    "best_checkpoint": str(CHECKPOINT_DIR / "best.pt"),
    "last_checkpoint": str(CHECKPOINT_DIR / "last.pt"),
    "last_preval_checkpoint": str(CHECKPOINT_DIR / "last_preval.pt"),
    "elapsed_minutes_this_session": elapsed / 60.0,
    "final_epoch": history[-1]["epoch"] if history else None,
    "config": CONFIG,
}

best_metrics_path = METRICS_DIR / "best_metrics.json"
best_metrics_path.write_text(json.dumps(best_metrics, indent=2), encoding="utf-8")

print("Training complete.")
print(f"Elapsed this session: {elapsed / 60:.2f} min")
print("Best lane F1:", best_lane_f1)
print("Best checkpoint:", CHECKPOINT_DIR / "best.pt")
print("Last checkpoint:", CHECKPOINT_DIR / "last.pt")
print("Last pre-validation checkpoint:", CHECKPOINT_DIR / "last_preval.pt")

In [ ]:
if OUTPUT_EXPORT_ROOT.exists():
    shutil.rmtree(OUTPUT_EXPORT_ROOT)

for p in [
    OUTPUT_EXPORT_ROOT / "checkpoints",
    OUTPUT_EXPORT_ROOT / "metrics",
    OUTPUT_EXPORT_ROOT / "configs",
]:
    p.mkdir(parents=True, exist_ok=True)

required_checkpoint_files = [
    CHECKPOINT_DIR / "best.pt",
    CHECKPOINT_DIR / "last.pt",
]

for ckpt_file in required_checkpoint_files:
    if not ckpt_file.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_file}")

shutil.copy2(CHECKPOINT_DIR / "best.pt", OUTPUT_EXPORT_ROOT / "checkpoints" / "best.pt")
shutil.copy2(CHECKPOINT_DIR / "last.pt", OUTPUT_EXPORT_ROOT / "checkpoints" / "last.pt")

for p in sorted(CHECKPOINT_DIR.glob("epoch_*.pt")):
    shutil.copy2(p, OUTPUT_EXPORT_ROOT / "checkpoints" / p.name)

shutil.copy2(METRICS_DIR / "history.json", OUTPUT_EXPORT_ROOT / "metrics" / "history.json")
shutil.copy2(METRICS_DIR / "best_metrics.json", OUTPUT_EXPORT_ROOT / "metrics" / "best_metrics.json")

shutil.copy2(
    train_config_path,
    OUTPUT_EXPORT_ROOT / "configs" / "stage1_drive_yolo_nano_384_train_config.json",
)

readme = f"""
# Drive Stage 1 YOLO-S+ Output

This dataset contains checkpoints and metrics for Drive Stage 1 continuous training.

## Architecture

Drive-YOLO-S+

Architecture upgrades:

- P1 high-resolution branch
- SPPF after P5
- BiFPN-lite weighted multi-scale neck
- Road segmentation head
- Lane segmentation head
- Edge extraction head

## Contents

- checkpoints/best.pt
- checkpoints/last.pt
- checkpoints/epoch_XXX.pt
- metrics/history.json
- metrics/best_metrics.json
- configs/stage1_drive_yolo_nano_384_train_config.json

## Best Metric

Best validation lane F1:

{best_lane_f1}

## Inputs Used

Original BDD100K:

/kaggle/input/bdd-100k

Processed masks:

/kaggle/input/drive-bdd100k-processed/drive-stage1-bdd100k-processed

## Training Notes

This is the continuous full-training notebook version.

Recommended default:

- epochs = 50
- batch_size = 4
- grad_accum_steps = 2
- effective batch size = 8
- AMP enabled
"""

(OUTPUT_EXPORT_ROOT / "README.md").write_text(readme.strip() + "\n", encoding="utf-8")

metadata = {
    "title": "Drive Stage 1 YOLO-S Plus Output",
    "id": "pandapiyush086/drive-stage1-yolo-s-plus-output",
    "licenses": [{"name": "CC0-1.0"}],
}

(OUTPUT_EXPORT_ROOT / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Packaged training output:", OUTPUT_EXPORT_ROOT)
!du -sh /kaggle/working/drive-stage1-yolo-s-plus-output
!find /kaggle/working/drive-stage1-yolo-s-plus-output -maxdepth 3 -type f | sort